# Requisito 1 Ingesta
Descargamos los 7 dias de datos AIS (NOAA, 2023-06-01 a 06-07) al Volume, los descomprimimos y losleemos como un unico DataFrame con esquema explicito. Tres pasos: **descarga + verificacion**, **lecturatipada** y una **comprobacion** rapida de que la ingesta quedo completa.

## Descarga, verificacion e ingesta

Por cada URL: descarga con reintentos (`download_with_retries`), verifica que el zip no este corrupto
(`verify_zip_integrity`), extrae los CSV al Volume y borra el zip. Si una descarga o la integridad
falla, se detiene con error en vez de continuar con datos incompletos.

In [0]:
import os
import time
import zipfile
import requests
import glob

volume_path = "/Volumes/proyecto_datos/default/data/ais_data"
os.makedirs(volume_path, exist_ok=True)

urls = [
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_01.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_02.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_03.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_04.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_05.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_06.zip",
    "https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_07.zip"
]

# Verificar si los 7 CSV ya existen en el Volume; si es asi, saltar descarga
existing_csvs = sorted(glob.glob(os.path.join(volume_path, "*.csv")))
if len(existing_csvs) >= 7:
    print(f"Se encontraron {len(existing_csvs)} archivos CSV ya presentes en {volume_path}.")
    print("Omitiendo descarga y extraccion. Los datos ya estan listos.")
else:
    def download_with_retries(url, dest_path, max_retries=3, delay=5):
        """Descarga un archivo con reintentos automaticos."""
        for attempt in range(1, max_retries + 1):
            try:
                print(f"Downloading {os.path.basename(dest_path)} (Attempt {attempt}/{max_retries})...")
                response = requests.get(url, stream=True, timeout=60)
                response.raise_for_status()
                
                with open(dest_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                return True
            except Exception as e:
                print(f"Attempt {attempt} failed: {e}")
                if attempt < max_retries:
                    time.sleep(delay)
                else:
                    raise RuntimeError(f"Failed to download {url} after {max_retries} attempts.")

    def verify_zip_integrity(zip_filepath):
        """Verifica que el zip sea valido y no este corrupto."""
        print(f"Verifying integrity of {os.path.basename(zip_filepath)}...")
        if not zipfile.is_zipfile(zip_filepath):
            return False
        
        with zipfile.ZipFile(zip_filepath, 'r') as zf:
            # testzip devuelve None si no hay archivos danados
            bad_file = zf.testzip()
            return bad_file is None

    # Logica principal de ingesta
    for url in urls:
        zip_name = url.split("/")[-1]
        zip_filepath = os.path.join(volume_path, zip_name)
        
        # Descargar con reintentos
        download_with_retries(url, zip_filepath, max_retries=3)
        
        # Verificar integridad
        if not verify_zip_integrity(zip_filepath):
            raise ValueError(f"Integrity check failed for {zip_name}. The file is corrupted.")
        print(f"Integrity verified for {zip_name}.")
        
        # 3. Extraer al Volume
        print(f"Extracting {zip_name}...")
        with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
            zip_ref.extractall(volume_path)
            
        os.remove(zip_filepath)

    print("All files downloaded, verified, and extracted successfully.")

Se encontraron 7 archivos CSV ya presentes en /Volumes/proyecto_datos/default/data/ais_data.
Omitiendo descarga y extraccion. Los datos ya estan listos.


## Lectura con esquema explicito

Leemos todos los CSV con un esquema declarado (`ais_schema`) en vez de inferirlo: garantiza los tipos
correctos (p. ej. `BaseDateTime` como `timestamp`, `LAT`/`LON` como `double`) y evita una pasada extra
de inferencia sobre 60 M de filas.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

BASE = "/Volumes/proyecto_datos/default/data/ais_data"

ais_schema = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG double,
    COG double,
    Heading double,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType int,
    Status int,
    Length double,
    Width double,
    Draft double,
    Cargo string,
    TransceiverClass string
"""

df_ais = (
    spark.read
    .option("header", "true")
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(ais_schema)
    .csv(f"{BASE}/*.csv")
)

## Comprobacion de la ingesta

Una vista de 10 filas para inspeccionar la estructura y el conteo total de registros, que confirma que
los 7 dias se cargaron (~60.5 M de posiciones).

In [0]:
display(df_ais.limit(10))

MMSI,BaseDateTime,LAT,LON,SOG,COG,Heading,VesselName,IMO,CallSign,VesselType,Status,Length,Width,Draft,Cargo,TransceiverClass
367138690,2023-06-01T00:00:01.000Z,30.28335,-91.22283,0.0,273.0,511.0,ST FRANCISVILLE,null,WDD3968,60,0,43.0,18.0,null,60,A
368036960,2023-06-01T00:00:01.000Z,40.49818,-80.07596,3.3,118.6,131.0,GALE R RHODES,IMO8199788,WDK2590,0,12,20.0,7.0,2.6,0,A
368095340,2023-06-01T00:00:01.000Z,29.99678,-93.94959,0.0,309.8,511.0,GAMBLER,null,WDK8562,52,15,19.0,8.0,0.0,52,A
248301000,2023-06-01T00:00:03.000Z,26.13342,-80.11909,0.0,260.5,86.0,ATLAS,null,9HB5599,37,5,31.0,8.0,3.0,37,A
367186410,2023-06-01T00:00:03.000Z,30.00762,-93.95571,5.7,164.0,164.0,BRIAN O DANIELS,null,WDD7438,31,12,26.0,9.0,null,31,A
367108820,2023-06-01T00:00:05.000Z,37.82833,-76.27936,0.1,235.2,511.0,TIDELANDS,IMO7309443,WX5026,30,0,61.0,9.0,null,30,A
367320740,2023-06-01T00:00:05.000Z,38.73458,-82.87265,4.7,350.3,349.0,TENNESSEE,IMO8836584,WDE2124,31,0,42.0,12.0,null,52,A
368101220,2023-06-01T00:00:03.000Z,32.6922,-117.14808,0.0,360.0,511.0,WILLIAM F,null,WDK9183,33,15,40.0,18.0,0.0,33,A
367529860,2023-06-01T00:00:08.000Z,29.61437,-95.00393,0.2,197.6,285.0,C R MCCASKILL,null,WDG3808,90,3,67.0,null,null,33,A
367558580,2023-06-01T00:00:09.000Z,27.3326,-82.54605,0.0,233.2,221.0,DAYS LIKE THIS,null,WDG6586,37,0,40.0,7.0,null,37,A


In [0]:
df_ais.count()

60533559

# Requisito 2 Exploracion y perfilamiento

**Objetivo.** Caracterizar el dataset AIS de la semana (7 dias) antes de responder preguntas de
negocio o diseñar el almacenamiento: cuanto volumen hay, como se comporta dia a dia, que tipos de
embarcación lo componen y, sobre todo, que problemas de calidad tiene, para definir las reglas de
limpieza de la Entrega 2.

El analisis se organiza en cuatro pasos:

1. **Preparacion** de columnas derivadas que se reutilizan en todo el requisito (`EventDate` y una
   marca de duplicados).
2. **Volumen** general y por dia (total de registros, buques unicos, actividad diaria).
3. **Perfilamiento** de embarcaciones: distribucion por tipo de buque (con velocidad media) y
   distribucion de dimensiones fisicas (eslora, manga, calado).
4. **Diagnostico de calidad**: una tabla unica que cuantifica cada regla de validacion sobre el
   total de filas. Es el insumo directo de la Entrega 2.

## Paso 1: Columnas derivadas de apoyo

Dos columnas que reutilizan los pasos siguientes:

- **`EventDate`**: la fecha (sin hora) de cada emision, para los cortes diarios.
- **`is_duplicate`**: marca de duplicado exacto. Definimos duplicado como una misma emision fisica,
  es decir, la combinacion `(MMSI, BaseDateTime, LAT, LON)` que aparece mas de una vez. En lugar de un
  `row_number()` sobre una ventana, cuyo orden dentro del grupo es arbitrario y confunde la lectura,
  contamos cuantas veces aparece cada combinacion (`count` sobre la particion) y marcamos como
  duplicada toda fila cuyo grupo tenga mas de un registro. Es equivalente en el conteo y autoexplicativo.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Fecha del evento (sin hora) para los cortes diarios
df_prepared = df_ais.withColumn("EventDate", F.to_date("BaseDateTime"))

# Marca de duplicado exacto: misma emision fisica repetida.
# Duplicado = combinacion (MMSI, BaseDateTime, LAT, LON) que aparece mas de una vez.
# Contamos el tamano de cada grupo y marcamos las filas cuyo grupo tiene mas de 1 registro.
dup_window = Window.partitionBy("MMSI", "BaseDateTime", "LAT", "LON")
df_flagged = df_prepared.withColumn(
    "is_duplicate",
    F.count("*").over(dup_window) > 1
)

# Conteo del hallazgo: filas marcadas como duplicadas (todas las copias del grupo)
# y numero de emisiones a eliminar (una copia por grupo se conserva).
dup_rows = df_flagged.filter(F.col("is_duplicate")).count()
dup_groups = (
    df_flagged.filter(F.col("is_duplicate"))
    .select("MMSI", "BaseDateTime", "LAT", "LON").distinct().count()
)
print(f"Filas en grupos duplicados: {dup_rows:,}")
print(f"Grupos duplicados (emisiones unicas afectadas): {dup_groups:,}")
print(f"Filas a eliminar para dejar una copia por grupo: {dup_rows - dup_groups:,}")

# Vista rapida de la estructura resultante
display(df_flagged.select("MMSI", "BaseDateTime", "LAT", "LON", "EventDate", "is_duplicate").limit(10))

Filas en grupos duplicados: 2,800
Grupos duplicados (emisiones unicas afectadas): 1,400
Filas a eliminar para dejar una copia por grupo: 1,400


MMSI,BaseDateTime,LAT,LON,EventDate,is_duplicate
0,2023-06-01T11:53:17.000Z,41.66808,-70.05943,2023-06-01,false
0,2023-06-01T12:22:27.000Z,41.66809,-70.05943,2023-06-01,false
0,2023-06-01T13:02:37.000Z,41.65678,-70.0596,2023-06-01,false
0,2023-06-01T13:55:17.000Z,41.60776,-70.20824,2023-06-01,false
0,2023-06-01T14:01:15.000Z,41.60397,-70.22738,2023-06-01,false
0,2023-06-01T14:14:17.000Z,41.60365,-70.27104,2023-06-01,false
0,2023-06-01T14:19:02.000Z,41.61182,-70.28339,2023-06-01,false
0,2023-06-01T14:33:00.000Z,41.63113,-70.29372,2023-06-01,false
0,2023-06-01T15:31:59.000Z,41.63949,-70.27562,2023-06-01,false
0,2023-06-01T15:46:59.000Z,41.6331,-70.27112,2023-06-01,false


## Paso 2: Volumen general y actividad diaria

Cuantificamos el tamano del dataset y su comportamiento en el tiempo:

- **Total de registros** y **buques unicos** (MMSI distintos) en la semana completa.
- **Posiciones y buques activos por dia**, para ver si la actividad es estable o tiene picos.

In [0]:
# Total de registros y buques unicos en la semana
total_records = df_ais.count()
total_unique_vessels = df_ais.select("MMSI").distinct().count()

print(f"Total de registros AIS: {total_records:,}")
print(f"Buques unicos (MMSI distintos): {total_unique_vessels:,}")

# Posiciones y buques unicos por dia
daily_summary = (
    df_flagged
    .groupBy("EventDate")
    .agg(
        F.count("*").alias("Total_Positions"),
        F.countDistinct("MMSI").alias("Unique_Vessels")
    )
    .orderBy("EventDate")
)

display(daily_summary)

Total de registros AIS: 60,533,559
Buques unicos (MMSI distintos): 31,871


EventDate,Total_Positions,Unique_Vessels
2023-06-01,8808904,20448
2023-06-02,9052241,21153
2023-06-03,8036348,20505
2023-06-04,8522645,19720
2023-06-05,8613757,19615
2023-06-06,8587938,19717
2023-06-07,8911726,20086


## Paso 3: Perfilamiento de embarcaciones

Dos vistas del "quien" del dataset:

- **Distribucion por tipo de buque** (`VesselType`): numero de posiciones, buques unicos y velocidad
  media (`SOG`) por tipo, ordenado por volumen de posiciones. El catalogo de tipos AIS permite luego
  traducir cada codigo (por ejemplo 31 = remolque, 70 = carga, 80 = tanquero).
- **Distribucion de dimensiones fisicas** (eslora, manga, calado) via percentiles p25/p50/p75/p99,
  para conocer el rango de tamanos y detectar valores extremos.

In [0]:
# Distribucion por tipo de buque (todos los tipos, ordenados por volumen)
vessel_type_dist = (
    df_ais
    .groupBy("VesselType")
    .agg(
        F.count("*").alias("Position_Count"),
        F.countDistinct("MMSI").alias("Unique_Vessels"),
        F.round(F.avg("SOG"), 2).alias("Avg_SOG_Knots")
    )
    .orderBy(F.desc("Position_Count"))
)
display(vessel_type_dist)

# Distribucion de dimensiones fisicas (Length, Width, Draft)
# Cada arreglo trae los percentiles en orden: [p25, p50 (mediana), p75, p99]
size_metrics = df_ais.select(
    F.expr("percentile_approx(Length, array(0.25, 0.50, 0.75, 0.99))").alias("Length_p25_p50_p75_p99"),
    F.expr("percentile_approx(Width,  array(0.25, 0.50, 0.75, 0.99))").alias("Width_p25_p50_p75_p99"),
    F.expr("percentile_approx(Draft,  array(0.25, 0.50, 0.75, 0.99))").alias("Draft_p25_p50_p75_p99")
)
display(size_metrics)

VesselType,Position_Count,Unique_Vessels,Avg_SOG_Knots
31,16558120,3451,1.9
37,14476696,12934,1.64
60,4795209,1708,4.43
30,4010188,2192,2.47
36,3876264,4243,1.98
90,3848162,1532,2.03
70,3837926,1563,6.37
52,1921565,482,2.09
80,1789168,818,5.87
57,1323765,242,1.91


Length_p25_p50_p75_p99,Width_p25_p50_p75_p99,Draft_p25_p50_p75_p99
"List(15.0, 22.0, 38.0, 323.0)","List(5.0, 8.0, 11.0, 48.0)","List(2.4, 3.7, 7.3, 17.0)"


## Paso 4: Diagnostico de calidad de datos

Una sola pasada de agregacion condicional cuenta, para cada regla de validacion, cuantas filas la
incumplen y su porcentaje sobre el total. Reunir todas las reglas en un unico `select` evita recorrer
los 60 M de filas una vez por regla. Las reglas cubren:

- **Posiciones fuera de rango**: `LAT` fuera de [-90, 90] o `LON` fuera de [-180, 180], o exactamente 0.0.
- **Velocidades (`SOG`) anomalas**: > 60 nudos (fisicamente imposible para trafico comercial) o negativas.
- **MMSI anomalos**: identificadores que no son de 9 digitos o codigos de prueba (`000000000`).
- **Heading**: valor 511 ("no disponible" segun la norma ITU) y valores fuera de [0, 359].
- **Dimensiones invalidas**: `Length` o `Width` <= 0.
- **Duplicados**: la marca `is_duplicate` del Paso 1 (cuenta *todas* las filas de un grupo repetido, no solo las copias sobrantes).

In [0]:
quality_diagnostics = df_flagged.select(
    # Total de filas, para calcular porcentajes
    F.count("*").alias("total_rows"),

    # Posiciones fuera de rango (LAT valida: -90 a 90, LON valida: -180 a 180)
    F.sum(F.when((F.col("LAT") < -90) | (F.col("LAT") > 90) | (F.col("LAT") == 0.0), 1).otherwise(0)).alias("invalid_lat_count"),
    F.sum(F.when((F.col("LON") < -180) | (F.col("LON") > 180) | (F.col("LON") == 0.0), 1).otherwise(0)).alias("invalid_lon_count"),

    # Anomalias de velocidad (SOG). Mayor a 60 nudos es imposible para naves comerciales. SOG menor a 0 es invalido
    F.sum(F.when(F.col("SOG") > 60.0, 1).otherwise(0)).alias("impossible_speed_count"),
    F.sum(F.when(F.col("SOG") < 0.0, 1).otherwise(0)).alias("negative_speed_count"),

    # MMSI anomalos: deben ser identificadores de 9 digitos. 000000000 o longitudes distintas son invalidos
    F.sum(F.when((F.length(F.col("MMSI")) != 9) | (F.col("MMSI") == "000000000") | F.col("MMSI").rlike("^0+$"), 1).otherwise(0)).alias("anomalous_mmsi_count"),

    # Anomalias de rumbo (Heading valido: 0 a 359 grados. 511 indica "no disponible")
    F.sum(F.when(F.col("Heading") == 511.0, 1).otherwise(0)).alias("heading_not_available_count"),
    F.sum(F.when((F.col("Heading") < 0.0) | ((F.col("Heading") > 359.0) & (F.col("Heading") != 511.0)), 1).otherwise(0)).alias("invalid_heading_count"),

    # Dimensiones invalidas (Length <= 0 o Width <= 0)
    F.sum(F.when((F.col("Length") <= 0) | (F.col("Width") <= 0), 1).otherwise(0)).alias("invalid_dimensions_count"),

    # Duplicados exactos (marca del Paso 1)
    F.sum(F.when(F.col("is_duplicate"), 1).otherwise(0)).alias("duplicate_records_count")
)

# Reestructurar a una tabla legible: una fila por regla
diag_row = quality_diagnostics.collect()[0]
total = diag_row["total_rows"]

def pct(n):
    return (n / total) * 100

quality_summary = spark.createDataFrame([
    ("Latitud invalida (|LAT| > 90 o 0.0)", diag_row["invalid_lat_count"], pct(diag_row["invalid_lat_count"])),
    ("Longitud invalida (|LON| > 180 o 0.0)", diag_row["invalid_lon_count"], pct(diag_row["invalid_lon_count"])),
    ("Velocidad imposible (SOG > 60 nudos)", diag_row["impossible_speed_count"], pct(diag_row["impossible_speed_count"])),
    ("Velocidad negativa (SOG < 0)", diag_row["negative_speed_count"], pct(diag_row["negative_speed_count"])),
    ("MMSI anomalo (longitud != 9 o dummy)", diag_row["anomalous_mmsi_count"], pct(diag_row["anomalous_mmsi_count"])),
    ("Heading no disponible (511.0)", diag_row["heading_not_available_count"], pct(diag_row["heading_not_available_count"])),
    ("Heading fuera de rango", diag_row["invalid_heading_count"], pct(diag_row["invalid_heading_count"])),
    ("Dimensiones invalidas (Length/Width <= 0)", diag_row["invalid_dimensions_count"], pct(diag_row["invalid_dimensions_count"])),
    ("Duplicados (MMSI + Timestamp + Coordenadas)", diag_row["duplicate_records_count"], pct(diag_row["duplicate_records_count"]))
], ["Quality_Issue", "Affected_Rows", "Percentage_Of_Total"])

display(quality_summary)

Quality_Issue,Affected_Rows,Percentage_Of_Total
Latitud invalida (|LAT| > 90 o 0.0),0,0.0
Longitud invalida (|LON| > 180 o 0.0),0,0.0
Velocidad imposible (SOG > 60 nudos),160613,0.26532885667601336
Velocidad negativa (SOG < 0),0,0.0
MMSI anomalo (longitud != 9 o dummy),49897,0.08242865746585294
Heading no disponible (511.0),33535628,55.40005998986447
Heading fuera de rango,1061,0.0017527467697711283
Dimensiones invalidas (Length/Width <= 0),2284033,3.7731682024511395
Duplicados (MMSI + Timestamp + Coordenadas),2800,0.004625533416926634


## Paso 5: Auditoria de nulos (completitud por columna)
Hay que saber en que columnas se puede confiar. Contamos, en una sola pasada, cuantos valores nulos tiene cada una de las 17 columnas y su porcentaje sobre el total. Columnas con muchos nulos (p. ej. `IMO`, `Cargo`, `Draft`) no sirven como llaves ni como features sin imputacion. Columnas casi completas (`MMSI`, `BaseDateTime`, `LAT`, `LON`) son las confiables.

In [0]:
# Conteo de nulos por columna, en una sola pasada sobre los 60 M de filas.
# Se cuenta como nulo el valor NULL y, en columnas numericas, tambien NaN.
cols = df_ais.columns

null_exprs = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in cols
]
null_row = df_ais.select(null_exprs).collect()[0]
total = df_ais.count()

null_summary = spark.createDataFrame(
    [(c, int(null_row[c]), (null_row[c] / total) * 100) for c in cols],
    ["Columna", "Nulos", "Porcentaje_Nulos"]
).orderBy(F.desc("Nulos"))

display(null_summary)

Columna,Nulos,Porcentaje_Nulos
Draft,39000865,64.42850155233727
IMO,25891448,42.772056405935096
Status,19952397,32.9608853826024
Cargo,19898489,32.8718306485168
CallSign,10157915,16.78063402814297
Width,9219567,15.23050544574787
Length,3664725,6.0540385540523065
VesselType,209863,0.3466886855273122
VesselName,150685,0.2489280367605678
MMSI,0,0.0


## Paso 6: Consistencia de atributos estaticos por buque (MMSI)
Un mismo buque (`MMSI`) deberia reportar **siempre** el mismo tipo y las mismas dimensiones durante la semana, porque son atributos estaticos. Cuando un `MMSI` reporta mas de un `VesselType` o mas de una eslora/manga, hay una inconsistencia: reprogramacion del transpondedor, reutilizacion de MMSI entre barcos, o error de captura. El hallazgo decide si podemos confiar en los atributos estaticos o si hay que resolver conflictos, por ejemplo, tomar el valor mas frecuente por MMSI. Contamos cuantos MMSI presentan cada tipo de inconsistencia.

In [0]:
# Para cada MMSI, cuantos valores DISTINTOS toma cada atributo estatico
mmsi_consistency = (
    df_ais
    .groupBy("MMSI")
    .agg(
        F.countDistinct("VesselType").alias("distinct_types"),
        F.countDistinct("Length").alias("distinct_lengths"),
        F.countDistinct("Width").alias("distinct_widths"),
        F.countDistinct("VesselName").alias("distinct_names")
    )
)

resumen_consistencia = mmsi_consistency.select(
    F.count("*").alias("total_mmsi"),
    F.sum(F.when(F.col("distinct_types") > 1, 1).otherwise(0)).alias("mmsi_varios_tipos"),
    F.sum(F.when(F.col("distinct_lengths") > 1, 1).otherwise(0)).alias("mmsi_varias_esloras"),
    F.sum(F.when(F.col("distinct_widths") > 1, 1).otherwise(0)).alias("mmsi_varias_mangas"),
    F.sum(F.when(F.col("distinct_names") > 1, 1).otherwise(0)).alias("mmsi_varios_nombres")
).collect()[0]

tm = resumen_consistencia["total_mmsi"]

def pct(n):
    return (n / tm) * 100

consistency_summary = spark.createDataFrame([
    ("MMSI con mas de un VesselType", resumen_consistencia["mmsi_varios_tipos"], pct(resumen_consistencia["mmsi_varios_tipos"])),
    ("MMSI con mas de una eslora (Length)", resumen_consistencia["mmsi_varias_esloras"], pct(resumen_consistencia["mmsi_varias_esloras"])),
    ("MMSI con mas de una manga (Width)", resumen_consistencia["mmsi_varias_mangas"], pct(resumen_consistencia["mmsi_varias_mangas"])),
    ("MMSI con mas de un VesselName", resumen_consistencia["mmsi_varios_nombres"], pct(resumen_consistencia["mmsi_varios_nombres"])),
], ["Inconsistencia", "MMSI_Afectados", "Porcentaje_De_MMSI"])

print(f"Total de MMSI distintos: {tm:,}")
display(consistency_summary)

# Ejemplos concretos: los MMSI que mas tipos distintos reportan
display(
    mmsi_consistency
    .filter(F.col("distinct_types") > 1)
    .orderBy(F.desc("distinct_types"))
    .limit(10)
)

Total de MMSI distintos: 31,871


Inconsistencia,MMSI_Afectados,Porcentaje_De_MMSI
MMSI con mas de un VesselType,0,0.0
MMSI con mas de una eslora (Length),0,0.0
MMSI con mas de una manga (Width),0,0.0
MMSI con mas de un VesselName,0,0.0


MMSI,distinct_types,distinct_lengths,distinct_widths,distinct_names


## Paso 7: Anomalias adicionales (temporales y de catalogo)

Dos inconsistencias baratas de medir y valiosas:

- **Posiciones imposibles en el tiempo**: un mismo `(MMSI, BaseDateTime)` con **coordenadas distintas**
  significa que el buque estaria en dos lugares a la vez en el mismo instante. Es un conflicto distinto
  del duplicado exacto del Paso 1 (mismo instante, distinto lugar) y hay que resolverlo antes de calcular
  trayectorias o distancias.
- **`VesselType` fuera de catalogo**: los tipos AIS validos van de 0 a 99. Contamos las filas con
  `VesselType` nulo o fuera de ese rango (en los datos aparecen codigos como 107, 136, 200).

In [0]:
# Posiciones imposibles en el tiempo: mismo (MMSI, BaseDateTime) con coordenadas distintas
time_window = Window.partitionBy("MMSI", "BaseDateTime")
df_time = df_ais.withColumn("distinct_coords", F.size(F.collect_set(F.concat_ws(",", "LAT", "LON")).over(time_window)))
impossible_time = df_time.filter(F.col("distinct_coords") > 1).count()

# VesselType fuera del catalogo AIS (valido 0-99) o nulo
invalid_type = df_ais.filter(
    F.col("VesselType").isNull() | (F.col("VesselType") < 0) | (F.col("VesselType") > 99)
).count()

# Volumen de filas con MMSI = 0 (no atribuibles a un buque)
mmsi_cero = df_ais.filter((F.col("MMSI") == "0") | F.col("MMSI").rlike("^0+$")).count()

total = df_ais.count()
anomalias = spark.createDataFrame([
    ("Posiciones imposibles en el tiempo (mismo MMSI+instante, distinto lugar)", impossible_time, (impossible_time / total) * 100),
    ("VesselType fuera de catalogo (nulo o fuera de 0-99)", invalid_type, (invalid_type / total) * 100),
    ("MMSI = 0 (no atribuible a un buque)", mmsi_cero, (mmsi_cero / total) * 100),
], ["Anomalia", "Filas_Afectadas", "Porcentaje_Del_Total"])

display(anomalias)

Anomalia,Filas_Afectadas,Porcentaje_Del_Total
"Posiciones imposibles en el tiempo (mismo MMSI+instante, distinto lugar)",544,8.986750638600318E-4
VesselType fuera de catalogo (nulo o fuera de 0-99),215721,0.35636596222601086
MMSI = 0 (no atribuible a un buque),978,0.0016156327434836601


## Diagnostico y hallazgos principales

**1. Volumen general y comportamiento diario**

- Total de registros procesados: **60,533,559** posiciones.
- Buques unicos en la semana: **31,871** MMSI distintos.
- Actividad diaria muy consistente: entre **8.0 y 9.0 millones** de posiciones por dia, con un promedio cercano a **20,000** buques activos diariamente.

**2. Perfilamiento de embarcaciones y dimensiones**

- Tipos dominantes: el tipo **31 (Towing/Tug)** y el tipo **37 (Pleasure Craft)** suman mas de **31 millones** de posiciones. Los buques mercantes y tanqueros (tipos **70** y **80**) registran velocidades medias mucho mayores (**5.8 a 6.4 nudos**) que las embarcaciones menores (**1.6 a 1.9 nudos**).
- Distribucion de tamano: la mediana es **22 m** de eslora (Length), **8 m** de manga (Width) y **3.7 m** de calado (Draft). En el percentil 99 aparecen embarcaciones de hasta **323 m** de eslora y **17 m** de calado (portacontenedores y grandes tanqueros).

**3. Completitud por columna (Paso 5)**

El nucleo dinamico del mensaje AIS esta **100% completo**: `MMSI`, `BaseDateTime`, `LAT`, `LON`, `SOG`, `COG`, `Heading` y `TransceiverClass` no tienen nulos. Los nulos se concentran en los atributos estaticos/opcionales:

| Columna | % nulos | Implicacion para la Entrega 2 |
|---|---:|---|
| `Draft` | 64.4% | Casi inutilizable como feature sin imputacion; solo confiable para el subconjunto que lo reporta. |
| `IMO` | 42.8% | No sirve como llave universal de buque; usar `MMSI`. |
| `Status` | 33.0% | Estado de navegacion ausente en 1/3 de las filas. |
| `Cargo` | 32.9% | Ademas suele igualar a `VesselType`; no es tonelaje. |
| `CallSign` | 16.8% | Identificador secundario incompleto. |
| `Width` / `Length` | 15.2% / 6.1% | Dimensiones parcialmente ausentes; afecta perfiles de tamano. |

**4. Diagnostico de calidad y anomalias (Pasos 4, 6 y 7)**

| Regla / anomalia | Filas o MMSI | % | Lectura |
|---|---:|---:|---|
| Heading no disponible (511.0) | 33,535,628 | 55.40% | No es corrupcion: la norma ITU usa 511 cuando el barco no tiene girocompas o esta estatico. Tratar como categoria nula, no numerica. |
| Dimensiones invalidas (Length/Width <= 0) | 2,284,033 | 3.77% | Eslora o manga <= 0; filtrar o imputar. |
| VesselType fuera de catalogo (nulo o fuera de 0-99) | 215,721 | 0.356% | 209,863 nulos + codigos como 107/136/200 que no existen en el catalogo AIS. Normalizar o marcar como "desconocido". |
| Velocidad imposible (SOG > 60 nudos) | 160,613 | 0.265% | Picos de ruido del GPS; descartar. |
| MMSI anomalo (longitud != 9 o dummy) | 49,897 | 0.082% | Identificadores no validos. |
| MMSI = 0 (no atribuible a un buque) | 978 | 0.0016% | Subconjunto de los MMSI anomalos; sin buque asociado. |
| Posiciones imposibles en el tiempo (mismo MMSI+instante, distinto lugar) | 544 | 0.0009% | Un buque no puede estar en dos lugares a la vez; resolver antes de calcular trayectorias. |
| Duplicados exactos (MMSI + Timestamp + Coordenadas) | 2,800 | 0.0046% | Todas las filas de grupos repetidos: ~1,400 pares, asi que basta eliminar 1,400 copias sobrantes. |
| Coordenadas fuera de rango | 0 | 0.00% | Excelente calidad geometrica: sin LAT/LON fuera de rango global. |

**5. Consistencia de atributos estaticos por MMSI (Paso 6): sin conflictos**

Contra lo que suele ocurrir en feeds AIS, en esta semana **ningun MMSI** reporta mas de un `VesselType`, ni mas de una eslora, manga o nombre (0 de 31,871 en las cuatro pruebas). Es un resultado **positivo y verificado**: los atributos estaticos son internamente consistentes por buque, asi que en la Entrega 2 se pueden tomar directamente (no hace falta resolver el valor "correcto" por MMSI). El unico problema de `VesselType` es de **catalogo** (nulos/codigos invalidos), no de **inconsistencia** entre emisiones del mismo buque.

## Conclusion

El dataset tiene **alta integridad estructural**: el nucleo dinamico del mensaje (posicion, tiempo, velocidad, rumbo) esta **100% completo y sin coordenadas fuera de rango**, y los atributos estaticos son **consistentes por buque** (ningun MMSI cambia de tipo o dimensiones). Los problemas se concentran en columnas opcionales muy incompletas y en unos pocos valores centinela/ruido.

Reglas de limpieza para la Entrega 2, ordenadas por impacto:

1. **Heading = 511** (55.4%): tratar como categoria nula / no disponible, no como valor numerico.
2. **Columnas con muchos nulos** (`Draft` 64%, `IMO` 43%, `Status`/`Cargo` 33%): decidir por columna entre descartarla, imputarla o usarla solo en el subconjunto que la reporta. No usar `IMO` como llave; usar `MMSI`.
3. **Dimensiones <= 0** (3.8%): filtrar o imputar.
4. **VesselType fuera de catalogo** (0.36%): normalizar a un valor "desconocido" en vez de tratar codigos invalidos como categorias reales.
5. **Velocidades > 60 nudos** (0.27%): descartar por ruido de GPS.
6. **MMSI no validos / = 0, duplicados exactos y posiciones imposibles en el tiempo**: eliminar; son volumen bajo pero rompen los calculos de trayectoria.

Todas las decisiones se justifican con la evidencia cuantitativa de los Pasos 4 a 7, no con supuestos.

# Requisito 3 Preguntas de negocio

#### a. ¿Cuántos buques distintos transmitieron cada día? Comparen el conteo exacto con approx_count_distinct y argumenten cuál usarían en producción.

In [0]:
unique_ships_day = (
    df_ais.withColumn("day", F.to_date("BaseDateTime"))
    .groupBy("day")
    .agg(F.countDistinct("MMSI").alias("unique_vessels"))
)
display(unique_ships_day)
unique_ships_day.explain("formatted")

day,unique_vessels
2023-06-04,19720
2023-06-05,19615
2023-06-02,21153
2023-06-06,19717
2023-06-01,20448
2023-06-07,20086
2023-06-03,20505


== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonGroupingAgg (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonGroupingAgg (8)
                        +- PhotonShuffleExchangeSource (7)
                           +- PhotonShuffleMapStage (6)
                              +- PhotonShuffleExchangeSink (5)
                                 +- PhotonGroupingAgg (4)
                                    +- PhotonProject (3)
                                       +- PhotonRowToColumnar (2)
                                          +- Scan csv  (1)


(1) Scan csv 
Output [2]: [MMSI#14600, BaseDateTime#14601]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/default/data/ais_data/AIS_2023_06_01.csv, ... 6 entries]
ReadSchema: 

In [0]:
approx_unique_ships_day = (
    df_ais.withColumn("day", F.to_date("BaseDateTime"))
    .groupBy("day")
    .agg(F.approx_count_distinct("MMSI").alias("unique_vessels"))
)
display(approx_unique_ships_day)
approx_unique_ships_day.explain("formatted")

day,unique_vessels
2023-06-02,23182
2023-06-07,21184
2023-06-01,21782
2023-06-05,20358
2023-06-06,20288
2023-06-04,20275
2023-06-03,20877


== Physical Plan ==
AdaptiveSparkPlan (11)
+- == Initial Plan ==
   PhotonResultStage (10)
   +- PhotonColumnarToRow (9)
      +- PhotonGroupingAgg (8)
         +- PhotonShuffleExchangeSource (7)
            +- PhotonShuffleMapStage (6)
               +- PhotonShuffleExchangeSink (5)
                  +- PhotonGroupingAgg (4)
                     +- PhotonProject (3)
                        +- PhotonRowToColumnar (2)
                           +- Scan csv  (1)


(1) Scan csv 
Output [2]: [MMSI#14600, BaseDateTime#14601]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/default/data/ais_data/AIS_2023_06_01.csv, ... 6 entries]
ReadSchema: struct<MMSI:string,BaseDateTime:timestamp>

(2) PhotonRowToColumnar
Input [2]: [MMSI#14600, BaseDateTime#14601]

(3) PhotonProject
Input [2]: [MMSI#14600, BaseDateTime#14601]
Arguments: [MMSI#14600, cast(BaseDateTime#14601 as date) AS day#15639]

(4) PhotonGroupingAgg
Input [2]: [MMSI#14600, day#15639]
Arguments: [day#15639], [par

El conteo difiere por poco para los dias. En produccion se usa el valor aproximado porque la importancia de este dato no justifica el precio de computo por un valor exacto. La aproximacion es suficiente para un reporte general.

#### b. ¿Qué tipos de buque generan más tráfico? Top 10 por número de posiciones, con velocidad media por tipo (usen el catálogo de tipos).

In [0]:
rank_vessel_type = (
    df_ais.groupBy("VesselType")
    .agg(
        F.count("*").alias("count"),
        F.avg("SOG").alias("average_speed")
    )
    .orderBy(F.desc("count"))
    .limit(10)
)

display(rank_vessel_type)
rank_vessel_type.explain("formatted")

VesselType,count,average_speed
31,16558120,1.8987526240930621
37,14476696,1.6422352103023863
60,4795209,4.433895873989091
30,4010188,2.466139193474545
36,3876264,1.9792365019505584
90,3848162,2.0281950188159845
70,3837926,6.369827792410587
52,1921565,2.093070023651827
80,1789168,5.874565719933229
57,1323765,1.9109420478708936


== Physical Plan ==
AdaptiveSparkPlan (15)
+- == Initial Plan ==
   PhotonResultStage (14)
   +- PhotonColumnarToRow (13)
      +- PhotonTopK (12)
         +- PhotonShuffleExchangeSource (11)
            +- PhotonShuffleMapStage (10)
               +- PhotonShuffleExchangeSink (9)
                  +- PhotonTopK (8)
                     +- PhotonGroupingAgg (7)
                        +- PhotonShuffleExchangeSource (6)
                           +- PhotonShuffleMapStage (5)
                              +- PhotonShuffleExchangeSink (4)
                                 +- PhotonGroupingAgg (3)
                                    +- PhotonRowToColumnar (2)
                                       +- Scan csv  (1)


(1) Scan csv 
Output [2]: [SOG#14604, VesselType#14610]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/default/data/ais_data/AIS_2023_06_01.csv, ... 6 entries]
ReadSchema: struct<SOG:double,VesselType:int>

(2) PhotonRowToColumnar
Input [2]: [SOG#14604,

**Respuesta.** Los tipos que generan más tráfico (por número de posiciones) son el **31 (Towing)**
con 16.5 M y el **37 (Pleasure Craft / remolcadores menores)** con 14.5 M; juntos concentran más de
la mitad de las emisiones. En velocidad media se ve el patrón esperado: los tipos de carga y tanqueros
—**70 (Cargo, 6.37 nudos)** y **80 (Tanker, 5.87 nudos)**— navegan sustancialmente más rápido que las
embarcaciones costeras y de servicio, que rondan 1.6–2.5 nudos. Es decir, el mayor **volumen** de
mensajes lo aportan embarcaciones lentas y muy numerosas, mientras que la mayor **velocidad** la
aportan los buques mercantes de línea, menos numerosos pero de tránsito continuo.

#### c. ¿Qué 10 buques recorrieron más distancia durante la semana? (orden por buque y tiempo, distancia entre posiciones consecutivas con haversine).

In [0]:
vessel_window = Window.partitionBy("MMSI").orderBy("BaseDateTime")
df = (
    df_ais.withColumn("prev_lat", F.lag("LAT").over(vessel_window))
    .withColumn("prev_lon", F.lag("LON").over(vessel_window))
)

lat1 = F.radians(F.col("prev_lat"))
lon1 = F.radians(F.col("prev_lon"))
lat2 = F.radians(F.col("LAT"))
lon2 = F.radians(F.col("LON"))

dlat = lat2 - lat1
dlon = lon2 - lon1

a = F.sin(dlat / 2) ** 2 + F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2) ** 2

r_km = 6371.0

distance_km = F.when(
    F.col("prev_lat").isNotNull() & F.col("prev_lon").isNotNull(),
    2 * r_km * F.asin(F.sqrt(a))
).otherwise(0.0)

df = df.withColumn("distance", distance_km)

top_10_distance_vessels = (
    df.groupBy("MMSI")
    .agg(F.sum("distance").alias("total_distance_km"))
    .orderBy(F.desc("total_distance_km"))
    .limit(10)
)

display(top_10_distance_vessels)
top_10_distance_vessels.explain("formatted")


MMSI,total_distance_km
211002010,7931112.219051635
368124160,7361374.945480704
367373000,5724082.305732174
338333275,5065029.783610111
374158000,3027501.8058918915
338561000,1613419.3413086575
360000000,1155988.0831470503
338479554,852140.4119425962
367530470,317418.18156711175
338182085,249765.17639226164


== Physical Plan ==
AdaptiveSparkPlan (17)
+- == Initial Plan ==
   PhotonResultStage (16)
   +- PhotonColumnarToRow (15)
      +- PhotonTopK (14)
         +- PhotonShuffleExchangeSource (13)
            +- PhotonShuffleMapStage (12)
               +- PhotonShuffleExchangeSink (11)
                  +- PhotonTopK (10)
                     +- PhotonGroupingAgg (9)
                        +- PhotonProject (8)
                           +- PhotonWindow (7)
                              +- PhotonSort (6)
                                 +- PhotonShuffleExchangeSource (5)
                                    +- PhotonShuffleMapStage (4)
                                       +- PhotonShuffleExchangeSink (3)
                                          +- PhotonRowToColumnar (2)
                                             +- Scan csv  (1)


(1) Scan csv 
Output [4]: [MMSI#14600, BaseDateTime#14601, LAT#14602, LON#14603]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/de

##### Jusitificacion
**Puntos claves del plan**:
+ Column Pruning
+ hashpartitioning(MMSI) y ordenamiento local
+ Top-K eficiente

La consulta se hizo de esta manera porque usa expresiones nativas para calcular la formula de Haversine y se aplico Window.partitionBy("MMSI").orderBy("BaseDateTime") junto con un limit(10). El plan de ejecucion confirma que el nodo Scan csv aplico column pruning leyendo solo 4 de las 17 columnas (MMSI, BaseDateTime, LAT, LON), reduciendo la I/O desde el origen. El shuffle por `hashpartitioning(MMSI)` permite ordenar localmente los datos por tiempo para procesar el `lag()` con un marco de memoria pequeño, mientras que el nodo `PhotonTopK` filtra los 10 mejores registros en cada ejecutor antes de enviar datos por la red a SinglePartition. No se hizo con una UDF de Python ni trayendo todos los datos ordenados al driver porque habria generado una sobrecarga masiva de serializacion entre la JVM y Python, ademas de un cuello de botella de red al transferir millones de agregaciones intermedias sin filtrar previo al shuffle final.

#### d. ¿Dónde se concentra el tráfico? Top 10 celdas de una grilla espacial (H3 resolución 8, o su propia rejilla lat/lon) por número de posiciones, y cuáles de esas celdas corresponden a puertos del World Port Index.

In [0]:
# 1. Map each position to its H3 Resolution 8 cell ID
df_h3 = df_ais.withColumn(
    "h3_cell", 
    F.expr("h3_longlatash3string(LON, LAT, 8)")
)

# 2. Find Top 10 spatial cells by position count
top_10_cells = (
    df_h3
    .groupBy("h3_cell")
    .agg(F.count("*").alias("position_count"))
    .orderBy(F.desc("position_count"))
    .limit(10)
)

display(top_10_cells)
top_10_cells.explain("formatted")

h3_cell,position_count
8828d555a1fffff,235268
8828d54715fffff,184414
8829a411d7fffff,180100
8828d17b59fffff,123785
8829a19a35fffff,115457
88446e0359fffff,114645
8829127827fffff,109522
8844a13b51fffff,103934
8828d5476bfffff,101039
8829a411a9fffff,100848


== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonTopK (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonTopK (9)
                     +- PhotonGroupingAgg (8)
                        +- PhotonShuffleExchangeSource (7)
                           +- PhotonShuffleMapStage (6)
                              +- PhotonShuffleExchangeSink (5)
                                 +- PhotonGroupingAgg (4)
                                    +- PhotonProject (3)
                                       +- PhotonRowToColumnar (2)
                                          +- Scan csv  (1)


(1) Scan csv 
Output [2]: [LAT#14602, LON#14603]
Batched: false
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/default/data/ais_data/AIS_2023_06_01.csv, ... 6 entries]
ReadSchema: struct<LAT:double,LON:do

In [0]:
# Leer el archivo de puertos del Wordl Port Index
df_wpi = (
    spark.read
    .option("header", "true")
    .csv("/Volumes/proyecto_datos/default/data/UpdatedPub150.csv")
    .select(
        F.col("World Port Index Number").cast("double"),
        F.col("Main Port Name"),
        F.col("Country Code"),
        F.col("Latitude").cast("double"),
        F.col("Longitude").cast("double")
    )
)
display(df_wpi.limit(10))

World Port Index Number,Main Port Name,Country Code,Latitude,Longitude
7950.0,Maurer,United States,40.533333,-74.25
47620.0,Iharana,Madagascar,-13.35,50.0
47020.0,Chake Chake,Tanzania,-5.25,39.766667
47005.0,Mjimwema Terminal,Tanzania,-6.816667,39.366667
44804.0,Delta Terminal,Turkey,36.85,36.166667
50350.0,Cinta Oil Terminal,Indonesia,-5.5,106.233333
38330.0,Europa Point,Gibraltar,36.133333,-5.35
7010.0,New Harbor,United States,43.866667,-69.483333
12900.0,Dtse / Gegua Oil Terminal,Brazil,-22.816667,-43.15
48104.0,Sharmah,Saudi Arabia,27.933333,35.25


In [0]:
# Calcular valor H3 para cada puerto
df_wpi_h3 = df_wpi.withColumn(
    "h3_cell",
    F.expr("h3_longlatash3string(Longitude, Latitude, 8)")
)

# Join Top 10 celdas AIS con World Port Index
top_10_ports_matched = (
    top_10_cells
    .join(df_wpi_h3, on="h3_cell", how="left")
    .select(
        "h3_cell",
        "position_count",
        F.col("Main Port Name").alias("matched_port_name"),
        F.col("Country Code").alias("country")
    )
    .orderBy(F.desc("position_count"))
)

display(top_10_ports_matched)
top_10_ports_matched.explain("formatted")

h3_cell,position_count,matched_port_name,country
8828d555a1fffff,235268,null,null
8828d54715fffff,184414,null,null
8829a411d7fffff,180100,null,null
8828d17b59fffff,123785,null,null
8829a19a35fffff,115457,null,null
88446e0359fffff,114645,null,null
8829127827fffff,109522,null,null
8844a13b51fffff,103934,null,null
8828d5476bfffff,101039,null,null
8829a411a9fffff,100848,null,null


== Physical Plan ==
AdaptiveSparkPlan (26)
+- == Initial Plan ==
   PhotonResultStage (25)
   +- PhotonColumnarToRow (24)
      +- PhotonSort (23)
         +- PhotonProject (22)
            +- PhotonBroadcastHashJoin LeftOuter (21)
               :- PhotonTopK (13)
               :  +- PhotonShuffleExchangeSource (12)
               :     +- PhotonShuffleMapStage (11)
               :        +- PhotonShuffleExchangeSink (10)
               :           +- PhotonTopK (9)
               :              +- PhotonGroupingAgg (8)
               :                 +- PhotonShuffleExchangeSource (7)
               :                    +- PhotonShuffleMapStage (6)
               :                       +- PhotonShuffleExchangeSink (5)
               :                          +- PhotonGroupingAgg (4)
               :                             +- PhotonProject (3)
               :                                +- PhotonRowToColumnar (2)
               :                                   +- Scan

Se encontro que para las posiciones mas transitadas no hay ningun puerto. El enlace directo por ID de celda fue demasiado estricto a esta resolución.

Esto podria explicarse en primer lugar porque la coordenada del puerto en el World Port Index es un punto arbitrario en tierra o sobre el muelle. Al asignarle una celda, esta no suele coincidir exactamente con la celda sobre el agua donde los barcos transmiten sus señales AIS. Ademas, a resolución 8, cada celda cubre solo ~0.73 km^2. Un barco transitando en la entrada del puerto estará a solo unos cientos de metros de la coordenada oficial, pero en una celda vecina.




##### Justificación (d)

La consulta se hizo de esta manera porque el conteo de posiciones por zona es una **agregación por
clave** (`groupBy(h3_cell).count()`) sobre la que solo se necesita el **Top 10**: el plan resuelve
esto con una agregación parcial en cada ejecutor, un único `hashpartitioning(h3_cell)` y un `TopK`
que recorta a 10 filas *antes* del `shuffle` final, así que por la red viajan decenas de filas y no
millones. Se eligió la grilla **H3 resolución 8** en vez de redondear lat/lon porque H3 da celdas de
área casi constante (~0.73 km²) y vecindad regular, apropiadas para "sectores marítimos" comparables
entre sí; el mapeo `h3_longlatash3string(LON, LAT, 8)` es una expresión nativa (sin UDF de Python),
por lo que se ejecuta dentro de Photon sin costo de serialización JVM↔Python.

El cruce con el World Port Index se hizo con un **`join` por igualdad de `h3_cell`** (no un
producto cartesiano de distancias) porque comparar cada celda contra cada puerto sería O(celdas ×
puertos); al llevar ambos lados a la misma clave H3, el `join` es una simple igualdad y, como el WPI
es una tabla pequeña, Spark puede difundirla (*broadcast*) y evitar un `shuffle` del lado grande.

**Resultado y su lectura:** el `join` exacto devolvió `null` para todas las celdas más transitadas.
No es un error del código sino una consecuencia de la resolución: a res 8 cada celda mide ~0.73 km²,
y la coordenada "oficial" de un puerto en el WPI es un punto en el muelle o en tierra, que cae en una
celda distinta a la que ocupan los barcos fondeados/transitando a unos cientos de metros. Es decir,
el emparejamiento por **ID de celda idéntico** es demasiado estricto para este propósito. No se
resolvió aquí con una búsqueda por distancia (vecindad `h3_grid_disk` o *range join* por radio)
porque eso corresponde al enriquecimiento de la Entrega 2; para la Entrega 1 basta con **cuantificar
la concentración de tráfico** (que sí se obtuvo) y documentar por qué el match directo no aplica.

#### e. ¿Qué proporción de los buques de la semana transmitió los 7 días? ¿Dónde están los "visitantes de un solo día"?

In [0]:
vessel_days = (
    df_ais
    .withColumn("date", F.to_date("BaseDateTime"))
    .groupBy("MMSI")
    .agg(F.countDistinct("date").alias("days_active"))
)

metrics = vessel_days.select(
    F.count("*").alias("total_unique_vessels"),
    F.sum(F.when(F.col("days_active") == 7, 1).otherwise(0)).alias("vessels_7_days"),
    F.sum(F.when(F.col("days_active") == 1, 1).otherwise(0)).alias("vessels_1_day")
).collect()[0]

total_vessels = metrics["total_unique_vessels"]
count_7_days = metrics["vessels_7_days"]
count_1_day = metrics["vessels_1_day"]

prop_7_days = (count_7_days / total_vessels) * 100
prop_1_day = (count_1_day / total_vessels) * 100

print(f"Total Unique Vessels: {total_vessels:,}")
print(f"Vessels active all 7 days: {count_7_days:,} ({prop_7_days:.2f}%)")
print(f"Single-day visitors (1 day only): {count_1_day:,} ({prop_1_day:.2f}%)")

vessel_days.explain("formatted")


Total Unique Vessels: 31,871
Vessels active all 7 days: 12,667 (39.74%)
Single-day visitors (1 day only): 5,996 (18.81%)
== Physical Plan ==
AdaptiveSparkPlan (16)
+- == Initial Plan ==
   PhotonResultStage (15)
   +- PhotonColumnarToRow (14)
      +- PhotonGroupingAgg (13)
         +- PhotonShuffleExchangeSource (12)
            +- PhotonShuffleMapStage (11)
               +- PhotonShuffleExchangeSink (10)
                  +- PhotonGroupingAgg (9)
                     +- PhotonGroupingAgg (8)
                        +- PhotonShuffleExchangeSource (7)
                           +- PhotonShuffleMapStage (6)
                              +- PhotonShuffleExchangeSink (5)
                                 +- PhotonGroupingAgg (4)
                                    +- PhotonProject (3)
                                       +- PhotonRowToColumnar (2)
                                          +- Scan csv  (1)


(1) Scan csv 
Output [2]: [MMSI#14600, BaseDateTime#14601]
Batched: false
Locati

El 39.74% del total de la flota, emitio señales durante los siete dias. El 18.81%, fueron visitantes de un solo dia.

In [0]:
# Filtrar MMSIs de un solo dia
single_day_mmsis = vessel_days.filter(F.col("days_active") == 1).select("MMSI")

# Mapear visitantes de un solo dia con su ubicacion espacial (H3 resolucion 8)
single_day_locations = (
    df_ais
    .join(single_day_mmsis, on="MMSI", how="inner")
    .withColumn("h3_cell", F.expr("h3_longlatash3string(LON, LAT, 8)"))
    .groupBy("h3_cell")
    .agg(
        F.count("*").alias("position_count"),
        F.countDistinct("MMSI").alias("unique_single_day_vessels"),
    )
    .orderBy(F.desc("unique_single_day_vessels"))
)

display(single_day_locations.limit(10))
single_day_locations.explain("formatted")

h3_cell,position_count,unique_single_day_vessels
882aa802cdfffff,266,49
8828d54715fffff,1319,47
882aa80213fffff,239,43
882aa80217fffff,811,42
8844a13b2dfffff,325,41
882aa80211fffff,273,41
882aa8021bfffff,245,39
8844a13b67fffff,179,38
882aa802e9fffff,510,35
8828d54711fffff,132,34


== Physical Plan ==
AdaptiveSparkPlan (41)
+- == Initial Plan ==
   PhotonResultStage (40)
   +- PhotonColumnarToRow (39)
      +- PhotonSort (38)
         +- PhotonShuffleExchangeSource (37)
            +- PhotonShuffleMapStage (36)
               +- PhotonShuffleExchangeSink (35)
                  +- PhotonGroupingAgg (34)
                     +- PhotonShuffleExchangeSource (33)
                        +- PhotonShuffleMapStage (32)
                           +- PhotonShuffleExchangeSink (31)
                              +- PhotonGroupingAgg (30)
                                 +- PhotonGroupingAgg (29)
                                    +- PhotonGroupingAgg (28)
                                       +- PhotonProject (27)
                                          +- PhotonShuffledHashJoin Inner (26)
                                             :- PhotonShuffleExchangeSource (6)
                                             :  +- PhotonShuffleMapStage (5)
                           

Respecto a la ubicacion espacial de los visitantes de un solo dia, las celdas revelan que la mayor concentracion se registra en la celda `882aa802cdfffff` con 49 embarcaciones distintas, seguida por la celda `8828d54715fffff` que registra 47 buques unicos.

##### Justificación (e)

**Proporción de buques por días activos.** Se calculó con `groupBy(MMSI).agg(countDistinct(date))`
para obtener los días distintos en que emitió cada buque, y luego una sola pasada de agregación
condicional (`sum(when(days_active == 7))` y `== 1`) para contar los dos extremos. El plan muestra la
doble agregación por `MMSI` con un `hashpartitioning(MMSI)`: primero se distinguen los pares
`(MMSI, date)` y después se cuentan por buque, todo en Spark nativo. No se usó una `Window` con
`collect_set` de fechas porque materializar el conjunto de fechas por buque es más caro en memoria que
un `countDistinct`, y aquí solo interesa el **número** de días, no el conjunto.

**Dónde están los visitantes de un solo día.** Se aisló la lista de MMSI con `days_active == 1` y se
volvió a unir contra `df_ais` (`join ... how="inner"`) para geolocalizar solo esas emisiones y
agregarlas por celda H3. La decisión clave es el **tipo de `join`**: la lista de MMSI de un solo día
es pequeña (unos ~6.000 identificadores), así que Spark puede **difundirla (*broadcast join*)** a cada
ejecutor y filtrar el DataFrame grande sin un `shuffle` de los 60 M de filas. Esto equivale a un
"semi-join" barato: se descarta de entrada la gran mayoría de posiciones (las de buques con más de un
día) y solo las restantes llegan al `groupBy(h3_cell)`. No se hizo con una subconsulta correlacionada
ni con un `filter` por `isin()` de miles de valores porque el `broadcast join` por clave es el patrón
que Spark optimiza para "filtrar una tabla grande con una lista pequeña".

> Verificación opcional: al añadir `.explain("formatted")` a `single_day_locations`, el nodo de
> `join` debería aparecer como *broadcast* (BroadcastHashJoin / PhotonBroadcastHashJoin); si por
> tamaño no se difundiera automáticamente, puede forzarse con `F.broadcast(single_day_mmsis)`.

# Requisito 4 Almacenamiento óptimo para un propósito

## Propósito de consulta y decisiones de almacenamiento

**Propósito de consulta:** monitoreo diario del operador portuario, que
consulta todas las posiciones de un **día específico** (`date`) dentro de un **sector marítimo**
delimitado por una celda de la grilla espacial (`h3_cell`, resolución 8).

**Decisiones (se justifican con la evidencia de los pasos B–F):**

| Decisión | Elección | Por qué |
|---|---|---|
| **Formato de archivo** | **Parquet** sobre CSV | CSV es texto sin tipos ni compresión. Para leer 4 columnas hay que leer la fila completa. Parquet guarda por columna con estadísticas min/max por bloque, habilitando *column pruning* y *predicate pushdown*. |
| **Formato de tabla** | **Delta** sobre solo Parquet | Delta = Parquet + un *transaction log*. El log guarda estadísticas por archivo y permite **data skipping**, `OPTIMIZE` (compactación) y `ZORDER` (co-localización). Con solo Parquet no hay log, así que no puede saltarse archivos por estadísticas ni compactar de forma transaccional. |
| **Layout: particionamiento** | `partitionBy("date")` | El propósito siempre filtra por fecha. Particionar por `date` crea 7 carpetas físicas. Un filtro por fecha lee solo 1 una carpeta (*partition pruning*), descartando 6/7 de los datos sin abrirlos. |
| **Layout: co-localización** | `OPTIMIZE ... ZORDER BY (h3_cell)` | Dentro de cada partición de fecha, el `ZORDER` agrupa las filas de una misma celda espacial en los mismos archivos. Con las estadísticas min/max del log, Delta salta los archivos que no contienen la celda buscada (*data skipping*), la segunda mitad del filtro de la consulta. |

No se elige CSV como formato final porque no permite *pushdown* ni *skipping* leyendo todo el archivo.<br>
No se elige solo Parquet como tabla porque, sin el log de Delta, no hay estadísticas por archivo para saltar datos, ni `OPTIMIZE`/`ZORDER`. La evidencia de abajo cuantifica ambas cosas.

## Paso A: Preparar columnas de partición (`date`) y grilla espacial (`h3_cell`)

Sobre el DataFrame de posiciones de los 7 días (`df_ais`) derivamos las dos columnas que usa la consulta

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, countDistinct, input_file_name

# Rutas base
CSV_PATH     = "/Volumes/proyecto_datos/default/data/ais_data/"
PARQUET_PATH = "/Volumes/proyecto_datos/default/data/ais_parquet/"
DELTA_TABLE  = "proyecto_datos.default.ais_positions_delta"

# date (particion) + h3_cell (sector espacial del proposito)
df_prepared = (
    df_ais
    .withColumn("date", to_date(col("BaseDateTime")))
    .withColumn("h3_cell", F.expr("h3_longlatash3string(LON, LAT, 8)"))
)

display(df_prepared.select("MMSI", "BaseDateTime", "date", "h3_cell").limit(5))

MMSI,BaseDateTime,date,h3_cell
368069540,2023-06-02T00:00:00.000Z,2023-06-02,8828c3c8a9fffff
538004494,2023-06-02T00:00:01.000Z,2023-06-02,88282235c9fffff
367614020,2023-06-02T00:00:01.000Z,2023-06-02,8828195807fffff
367372810,2023-06-02T00:00:02.000Z,2023-06-02,884450c5c1fffff
367148660,2023-06-02T00:00:05.000Z,2023-06-02,88464bb227fffff


## Paso B: Escribir las dos alternativas: Parquet y Delta

Escribimos el mismo DataFrame en los dos formatos que vamos a comparar, ambos particionados por `date`
para que la comparación sea justa. Mismo layout lógico, distinto motor de tabla.

In [0]:
# Baseline en Parquet (sin transaction log)
(df_prepared.write
    .format("parquet")
    .mode("overwrite")
    .partitionBy("date")
    .save(PARQUET_PATH)
)

# Tabla Delta (Parquet + transaction log) particionada por fecha
(df_prepared.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("date")
    .option("overwriteSchema", "true")
    .saveAsTable(DELTA_TABLE)
)

print("Parquet escrito en:", PARQUET_PATH)
print("Tabla Delta creada:", DELTA_TABLE)

Parquet escrito en: /Volumes/proyecto_datos/default/data/ais_parquet/
Tabla Delta creada: proyecto_datos.default.ais_positions_delta


## Paso C: Evidencia 1: bytes antes / después (CSV, Parquet, Delta)

Medimos el tamaño de cada representación. Para CSV y Parquet recorremos el directorio con
`dbutils.fs`. Para Delta leemos el tamaño autoritativo del *transaction log* con `DESCRIBE DETAIL`.

In [0]:
def dir_size_bytes(path):
    """Suma recursiva del tamano de todos los archivos bajo un path del Volume."""
    total = 0
    for f in dbutils.fs.ls(path):
        if f.isDir():
            total += dir_size_bytes(f.path)
        else:
            total += f.size
    return total

def gb(b):
    return b / (1024 ** 3)

# CSV y Parquet: tamano en disco por recorrido de directorio
size_csv = dir_size_bytes(CSV_PATH)
size_parquet = dir_size_bytes(PARQUET_PATH)

# Delta: tamano desde el log (sizeInBytes) + numFiles
detail = spark.sql(f"DESCRIBE DETAIL {DELTA_TABLE}").select("sizeInBytes", "numFiles").collect()[0]
size_delta = detail["sizeInBytes"]
delta_num_files = detail["numFiles"]

print(f"CSV     : {gb(size_csv):6.2f} GB")
print(f"Parquet : {gb(size_parquet):6.2f} GB   ({(1 - size_parquet/size_csv)*100:5.1f}% menos que CSV)")
print(f"Delta   : {gb(size_delta):6.2f} GB   ({(1 - size_delta/size_csv)*100:5.1f}% menos que CSV)  | archivos: {delta_num_files}")

CSV     :   6.04 GB
Parquet :   2.01 GB   ( 66.7% menos que CSV)
Delta   :   1.48 GB   ( 75.6% menos que CSV)  | archivos: 7


## Paso D: Evidencia 2: archivos escaneados por la consulta

Ejecutamos la consulta sobre cada fuente y recogemos **dos evidencias complementarias**:

1. **Plan de ejecución** (`explain("formatted")`): en el nodo `Scan` se leen los
   `PartitionFilters` (prueba del *partition pruning* por `date`) y las `DictionaryFilters` /
   `RequiredDataFilters` sobre `h3_cell` (el *data skipping* que Delta aplica con las estadísticas de su *transaction log*).

2. **Archivos con datos** (`_metadata.file_path`): número de archivos físicos que aportan filas al
   resultado. Menos archivos = mejor *pruning/skipping* para este patrón de acceso.
   
   (Nota: en Unity Catalog `input_file_name()` no está soportado. Se usa la columna `_metadata.file_path`.)

Ambos formatos aplican el mismo *partition pruning* por `date` (idéntico `PartitionFilters` en el
plan), así que descartan 6 de los 7 días. La diferencia aparece **dentro** del día: Parquet debe
abrir todos los *part-files* de esa partición, mientras que Delta usa las estadísticas por archivo
de su log para saltar los que no contienen la celda buscada.

In [0]:
target_date = "2023-06-01"
target_cell = "882aa802cdfffff" # celda con mas visitantes de un solo dia (Req 3e)

def purpose_query(df):
    """La consulta especificada: filtra por fecha (particion) y sector espacial (h3_cell)."""
    return df.filter((col("date") == target_date) & (col("h3_cell") == target_cell))

def files_and_rows(df, label, show_plan=False):
    q = purpose_query(df)
    n_files = q.withColumn("_file", col("_metadata.file_path")).select(countDistinct("_file")).collect()[0][0]
    n_rows  = q.count()
    print(f"{label:26s}: Archivos con datos: {n_files:4d} . Filas: {n_rows}")
    if show_plan:
        print(f"\n----- PLAN: {label} -----")
        q.explain("formatted")
        print()
    return n_files, n_rows

df_parquet = spark.read.format("parquet").load(PARQUET_PATH)
df_delta = spark.read.table(DELTA_TABLE)

print("=== Archivos escaneados por la consulta ===")
f_par, _ = files_and_rows(df_parquet, "Parquet (partition date)",   show_plan=True)
f_del_pre, _ = files_and_rows(df_delta,   "Delta ANTES de OPTIMIZE", show_plan=True)

=== Archivos escaneados por la consulta ===
Parquet (partition date)  : Archivos con datos:    6 . Filas: 242

----- PLAN: Parquet (partition date) -----
== Physical Plan ==
PhotonResultStage (3)
+- PhotonColumnarToRow (2)
   +- PhotonScan parquet  (1)


(1) PhotonScan parquet 
Output [19]: [MMSI#18455, BaseDateTime#18456, LAT#18457, LON#18458, SOG#18459, COG#18460, Heading#18461, VesselName#18462, IMO#18463, CallSign#18464, VesselType#18465, Status#18466, Length#18467, Width#18468, Draft#18469, Cargo#18470, TransceiverClass#18471, h3_cell#18472, date#18473]
DictionaryFilters: [(h3_cell#18472 = 882aa802cdfffff)]
Location: InMemoryFileIndex [dbfs:/Volumes/proyecto_datos/default/data/ais_parquet]
PartitionFilters: [isnotnull(date#18473), (date#18473 = 2023-06-01)]
ReadSchema: struct<MMSI:string,BaseDateTime:timestamp,LAT:double,LON:double,SOG:double,COG:double,Heading:double,VesselName:string,IMO:string,CallSign:string,VesselType:int,Status:int,Length:double,Width:double,Draft:double,Car

## Paso E: Aplicar `OPTIMIZE` y `ZORDER BY (h3_cell)`

`OPTIMIZE` reescribe los datos de cada partición y `ZORDER BY (h3_cell)` co-localiza en los mismos
archivos las filas de una misma celda espacial, para maximizar el *data skipping* de consultas por
zona. La fila de métricas de salida (`numFilesAdded` / `numFilesRemoved`) documenta la reescritura.

**Qué esperar con nuestros datos.** Nuestra tabla es pequeña por partición (7 archivos, uno por día).
`OPTIMIZE` no reduce el número de archivos aquí: al aplicar `ZORDER` la tabla pasa de 7 a 21
archivos, porque el reordenamiento por celda divide cada archivo diario en varios *Z-cubes*. La
compactación que baja el conteo de archivos solo ayuda cuando hay muchos archivos pequeños por
partición que no es nuestro caso. Por eso medimos el efecto sobre la **consulta**, no solo sobre el conteo de archivos.

In [0]:
%sql
-- Compactacion y co-localizacion espacial dentro de cada particion de fecha
OPTIMIZE proyecto_datos.default.ais_positions_delta
ZORDER BY (h3_cell);

path,metrics
,"List(21, 7, List(63471549, 82127467, 7.239809609523809E7, 21, 1520360018), List(210409463, 240171464, 2.2631319057142857E8, 7, 1584192334), 7, List(minCubeSize(107374182400), List(0, 0), List(7, 1584192334), 0, List(7, 1584192334), 7, null), null, 0, 1, 7, 0, false, 0, 0, 1790384680522, 1790384754386, 8, 7, null, List(0, 0), null, 19, 19, 124290, 0, null, null, 0)"


In [0]:
# Volver a medir tras OPTIMIZE/ZORDER
df_delta_opt = spark.read.table(DELTA_TABLE)

print("=== Archivos escaneados: efecto de OPTIMIZE + ZORDER ===")
f_del_post, _ = files_and_rows(df_delta_opt, "Delta DESPUES de OPTIMIZE", show_plan=True)

detail2 = spark.sql(f"DESCRIBE DETAIL {DELTA_TABLE}").select("numFiles", "sizeInBytes").collect()[0]
print()
print(f"Delta numFiles: {delta_num_files} (antes)  ->  {detail2['numFiles']} (despues de compactar)")
print("Archivos tocados por la consulta del proposito:")
print(f"  Parquet                 : {f_par}")
print(f"  Delta (antes OPTIMIZE)  : {f_del_pre}")
print(f"  Delta (despues ZORDER)  : {f_del_post}")

=== Archivos escaneados: efecto de OPTIMIZE + ZORDER ===
Delta DESPUES de OPTIMIZE : Archivos con datos:    1 . Filas: 242

----- PLAN: Delta DESPUES de OPTIMIZE -----
== Physical Plan ==
PhotonResultStage (4)
+- PhotonColumnarToRow (3)
   +- PhotonProject (2)
      +- PhotonScan parquet proyecto_datos.default.ais_positions_delta (1)


(1) PhotonScan parquet proyecto_datos.default.ais_positions_delta
Output [19]: [MMSI#19833, BaseDateTime#19834, LAT#19835, LON#19836, SOG#19837, COG#19838, Heading#19839, VesselName#19840, IMO#19841, CallSign#19842, VesselType#19843, Status#19844, Length#19845, Width#19846, Draft#19847, Cargo#19848, TransceiverClass#19849, h3_cell#19851, date#19850]
DictionaryFilters: [(h3_cell#19851 = 882aa802cdfffff)]
Location: PreparedDeltaFileIndex [s3://dbstorage-prod-f010s/uc/6ddea1a1-9973-40df-a29d-eb4dbd4bf962/4735a373-8e6d-431c-8096-841dd3b90846/__unitystorage/catalogs/a4f6ad21-10a4-4747-a7e5-a3692caa2e16/tables/cae1f65b-ff43-4b0d-92de-4ca97ca2647c]
PartitionFil

## Resumen comparativo (formato, bytes y archivos leídos)

Tabla única con las tres evidencias juntas para la consulta
(`date = 2023-06-01 AND h3_cell = 882aa802cdfffff`).

In [0]:
# Resumen: consolida bytes (Paso C) y archivos leidos por la consulta (Pasos D y E)
resumen = spark.createDataFrame(
    [
        ("CSV (crudo)", round(gb(size_csv), 2), None, None),
        ("Parquet (partition date)", round(gb(size_parquet), 2), int(f_par), None),
        ("Delta (antes OPTIMIZE)", round(gb(size_delta), 2), int(f_del_pre), int(delta_num_files)),
        ("Delta (despues ZORDER)", round(gb(size_delta), 2), int(f_del_post), int(detail2["numFiles"])),
    ],
    ["formato", "tamano_GB", "archivos_leidos_consulta", "archivos_totales_tabla"],
)

display(resumen)

formato,tamano_GB,archivos_leidos_consulta,archivos_totales_tabla
CSV (crudo),6.04,null,null
Parquet (partition date),2.01,6,null
Delta (antes OPTIMIZE),1.48,1,7
Delta (despues ZORDER),1.48,1,21


## Conclusión

El almacenamiento se resolvió con Delta particionado por `date` porque el propósito de consulta
filtra siempre por fecha y por sector espacial, y el layout ataca esas dos dimensiones. La evidencia
medida lo respalda:

1. **Bytes (Paso C):** la compresión columnar reduce el tamaño de forma drástica frente al CSV crudo:
   **CSV 6.04 GB -> Parquet 2.01 GB (−66.7%) -> Delta 1.48 GB (−75.6%)**.
2. **Archivos leídos por la consulta (Paso D):** con el mismo `PartitionFilters` por
   `date`, Parquet abre 6 archivos de la partición del día, mientras que Delta abre solo 1.
   Esa diferencia es el *data skipping* de Delta: las estadísticas por archivo de su
   *transaction log*, combinadas con las `DictionaryFilters`/`RequiredDataFilters` sobre `h3_cell`
   del plan, le permiten saltar los archivos que no contienen la celda buscada. Es la ganancia de
   Delta sobre Parquet, y ocurre sin necesidad de `OPTIMIZE`.
3. **Efecto de `OPTIMIZE` + `ZORDER` (Paso E):** a nuestra escala **no mejora la consulta** (Delta ya
   leía 1 solo archivo por partición) y de hecho sube el número de archivos de 7 a 21 por el
   reordenamiento en *Z-cubes*. `OPTIMIZE` es una herramienta para tablas con muchos archivos
   pequeños por partición. Con 7 días y un archivo por día no hay nada que compactar, así que su
   beneficio no se nota aqui. Se documenta como parte del análisis, no como una mejora observada.

**Decisión final:** *Delta particionado por `date`*. `ZORDER`/`OPTIMIZE` se mantienen en el pipeline porque escalarían con más días o escritura incremental, a pesar de que con 7 días su efecto sobre la consulta sea nulo.

# Requisito 5 Gobernanza: verificacion de comentarios y metadatos

Los comentarios (`COMMENT`) y metadatos (`TBLPROPERTIES`) sobre el catalogo, el esquema, la tabla,
sus columnas y el volumen se aplicaron en Unity Catalog. Como esos objetos viven en el metastore y no
en el repositorio Git, esta consulta deja evidencia en el notebook de que la gobernanza
esta aplicada, sin acceso directo al workspace.

`DESCRIBE EXTENDED` muestra el comentario de la tabla y sus `Table Properties`.

In [0]:
%sql
-- Comentario de tabla + propiedades (buscar 'Comment' y 'Table Properties' en la salida)
DESCRIBE EXTENDED proyecto_datos.default.ais_positions_delta;

col_name,data_type,comment
MMSI,string,null
BaseDateTime,timestamp,null
LAT,double,null
LON,double,null
SOG,double,null
COG,double,null
Heading,double,null
VesselName,string,null
IMO,string,null
CallSign,string,null
